# Bias & Fairness Analysis — NovaCred Credit Applications

**Course:** Data Ecosystems and Governance in Organizations (DEGO 2606)
**Institution:** Nova School of Business and Economics — MSc Business Analytics
**Role:** Data Scientist

---

| Item | Detail |
|---|---|
| Protected attributes | Gender · Age (derived from `date_of_birth`) |
| Outcome variable | `loan_approved` (historical decision outcome) |
| Analysis type | Audit of historical credit decisions — not a trained ML model |
| Reference date | 2024-01-01 (fixed for reproducibility) |
| Input dataset | Silver layer — `data/processed/cleaned_credit_applications.json` |

## Table of Contents

1. [Scope and Methodology](#scope)
2. [Data Loading and Preprocessing](#data)
3. [Baseline Disparity Analysis](#baseline)
4. [Conditional Fairness Analysis](#conditional)
5. [Pricing Fairness](#pricing)
6. [Proxy Discrimination Risk](#proxy)
7. [Intersectional Effects](#intersectional)
8. [Fairness Metrics Summary](#fairlearn)
9. [Governance Risk Assessment](#risk)
10. [Executive Summary](#executive)

<a id="scope"></a>
## 1. Scope and Methodology

This notebook constitutes the **Bias & Fairness** pillar of the NovaCred governance audit. The objective is to determine whether NovaCred's historical credit approval decisions exhibit statistically significant disparities along legally protected characteristics — and whether those disparities persist after controlling for legitimate financial risk factors.

**Protected attributes under analysis:**

| Attribute | Source field | Rationale |
|---|---|---|
| Gender | `applicant_info.gender` | Standardised to Male / Female / Unknown in Silver layer |
| Age | `applicant_info.date_of_birth` | Derived using fixed reference date; age group analysis follows ECOA conventions |

**Analytical framework:**

| Section | Method | Decision threshold |
|---|---|---|
| Baseline disparity | Approval rate comparison + χ² test | p < 0.05 · DI < 0.80 = regulatory breach |
| Conditional fairness | Logistic regression with financial controls | Gender coefficient p-value < 0.05 |
| Pricing fairness | Mann–Whitney U + OLS regression | p < 0.05 |
| Proxy discrimination | χ² (proxy ↔ attribute) + logistic regression (proxy → outcome) | Both conditions must hold simultaneously |
| Intersectional effects | Gender × Age group approval rates | DI < 0.80 per sub-group |
| Fairness metrics | Fairlearn `MetricFrame` | Demographic parity difference |

**Disparate Impact (DI) rule:** A DI ratio below **0.80** (the "four-fifths rule") constitutes a prima facie indicator of adverse impact under EEOC guidelines and analogous EU credit discrimination frameworks.



**Regulatory context — EU AI Act:**  
Credit scoring systems fall under **Annex III** of the EU AI Act as high-risk AI applications. Article 10 requires that training and evaluation data be subject to data governance practices that identify and address possible biases. This audit directly operationalises that requirement by testing whether historical decision outcomes reflect bias along protected characteristics, and by proposing governance controls proportionate to the identified risks.

**Note on Unknown gender:** Two records carry the value `Unknown` after cleaning. These records are excluded from all statistical tests — the group is too small to support reliable inference.

In [ ]:
import json
import os
import warnings
from collections import defaultdict
from datetime import date

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────────────────────
REF_DATE   = date(2024, 1, 1)   # fixed reference date — never use "today"
FIG_DIR    = os.path.join("..", "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────────
COLOUR_F   = "#e05c5c"   # Female
COLOUR_M   = "#4a90d9"   # Male
COLOUR_NEU = "#7f7f7f"   # Neutral / age groups
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})

<a id="data"></a>
## 2. Data Loading and Preprocessing

In [ ]:
# ── 2.1 Load Silver-layer dataset ─────────────────────────────────────────────
# Portable path resolution — works regardless of working directory
_candidates = [
    os.path.join("data", "processed", "cleaned_credit_applications.json"),
    os.path.join("..", "data", "processed", "cleaned_credit_applications.json"),
]
for _p in _candidates:
    if os.path.exists(_p):
        DATA_PATH = _p
        break
else:
    raise FileNotFoundError(
        "cleaned_credit_applications.json not found. "
        "Run 01_data_quality_assessment.ipynb first to generate the Silver-layer file."
    )

with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)

df_raw = pd.json_normalize(records)
print(f"Dataset : {DATA_PATH}")
print(f"Records : {len(df_raw)}")

In [ ]:
# ── 2.2 Silver-layer validation ───────────────────────────────────────────────
# Confirms we are operating on the cleaned dataset, not the raw Bronze layer.

def _pct(n, total): return f"{n/total*100:.1f}%"

checks = {
    "Record count == 500"          : len(df_raw) == 500,
    "No negative credit_history"   : pd.to_numeric(
                                        df_raw["financials.credit_history_months"],
                                        errors="coerce").fillna(0).ge(0).all(),
    "No negative savings_balance"  : pd.to_numeric(
                                        df_raw["financials.savings_balance"],
                                        errors="coerce").fillna(0).ge(0).all(),
    "Gender values standardised"   : set(df_raw["applicant_info.gender"].unique()
                                        ).issubset({"Male", "Female", "Unknown"}),
}

print("Silver-layer validation checks:")
all_pass = True
for desc, passed in checks.items():
    status   = "PASS" if passed else "FAIL"
    all_pass = all_pass and passed
    print(f"  [{status}] {desc}")

print()
if all_pass:
    print("All checks passed — Silver-layer dataset confirmed.")
else:
    print("WARNING: one or more checks failed. Verify the data pipeline before proceeding.")

In [ ]:
# ── 2.3 Feature engineering ───────────────────────────────────────────────────

df = df_raw.copy()

# Income: merge annual_income + annual_salary (schema inconsistency documented in DQ notebook §2.4)
_inc = pd.to_numeric(df["financials.annual_income"], errors="coerce")
_sal = pd.to_numeric(df["financials.annual_salary"],  errors="coerce")
df["income"] = _inc.combine_first(_sal)

# Age: derived from date_of_birth using fixed reference date
def _age(dob):
    try:
        d = date.fromisoformat(str(dob).strip())
        return (REF_DATE - d).days // 365
    except Exception:
        return None

df["age"] = df["applicant_info.date_of_birth"].apply(_age)

# Age groups (ECOA-aligned bins)
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 30, 40, 50, 120],
    labels=["<30", "30-39", "40-49", "50+"],
    right=False,
)

# ZIP prefix (first 3 digits) — used in proxy analysis
df["zip3"] = df["applicant_info.zip_code"].astype(str).str[:3].str.strip()

# Spending behaviour: pivot to per-record features
spend_rows = []
for r in records:
    row = {"_id": r["_id"]}
    for s in (r.get("spending_behavior") or []):
        key = "spend_" + s["category"].lower().replace(" ", "_")
        row[key] = float(s.get("amount", 0))
    spend_rows.append(row)

df_spend = pd.DataFrame(spend_rows).set_index("_id")
df = df.set_index("_id").join(df_spend, how="left").reset_index()

# Flagged spending: any high-risk category present (used in proxy §6.2)
flagged_cats = ["spend_gambling", "spend_alcohol", "spend_adult_entertainment"]
df["spend_flagged"] = df[[c for c in flagged_cats if c in df.columns]].notna().any(axis=1).astype(int)

# Convenience aliases
df["approved"]      = df["decision.loan_approved"].astype(int)
df["gender"]        = df["applicant_info.gender"]
df["interest_rate"] = df["decision.interest_rate"]
df["dti"]           = df["financials.debt_to_income"]
df["credit_hist"]   = df["financials.credit_history_months"]
df["savings"]       = df["financials.savings_balance"]

# Analysis set: exclude Unknown gender (n=2)
df_k = df[df["gender"].isin(["Male", "Female"])].copy()

print("Feature engineering complete:")
print(f"  Full dataset     : {len(df)} records")
print(f"  Analysis set     : {len(df_k)} records  "
      f"(Male={( df_k['gender']=='Male').sum()}, Female={(df_k['gender']=='Female').sum()})")
print(f"  Age nulls        : {df_k['age'].isna().sum()}")
print(f"  Income nulls     : {df_k['income'].isna().sum()}")
print(f"  credit_hist nulls: {df_k['credit_hist'].isna().sum()}")
print(f"  savings nulls    : {df_k['savings'].isna().sum()}")

<a id="baseline"></a>
## 3. Baseline Disparity Analysis

This section quantifies raw approval-rate disparities by gender and age group, applies the four-fifths (80%) disparate impact rule, and tests statistical significance via Pearson's chi-square test.
No financial controls are applied here — controlled analysis follows in Section 4.

### 3.1 Gender — Approval Disparity

In [ ]:
# ── 3.1 Gender approval rates ─────────────────────────────────────────────────
gen_agg = (df_k.groupby("gender")["approved"]
               .agg(approved_n="sum", total="count", approval_rate="mean")
               .assign(approval_rate=lambda x: x["approval_rate"].round(4)))

print("Approval Rates by Gender:")
print(gen_agg.to_string())

rate_f = gen_agg.loc["Female", "approval_rate"]
rate_m = gen_agg.loc["Male",   "approval_rate"]
di_gen = rate_f / rate_m

print(f"\n  Overall approval rate : {df_k['approved'].mean():.4f}")
print(f"  Female approval rate  : {rate_f:.4f}  ({rate_f:.1%})")
print(f"  Male approval rate    : {rate_m:.4f}  ({rate_m:.1%})")
print(f"  Gap (Male - Female)   : {rate_m - rate_f:.4f}  ({(rate_m - rate_f)*100:.1f} pp)")
print(f"\n  Disparate Impact Ratio (Female ÷ Male) : {di_gen:.4f}")
print(f"  80% Rule threshold: 0.80  →  {'FAIL — adverse impact indicated' if di_gen < 0.80 else 'PASS'}")

# Chi-square test
ct_gen = pd.crosstab(df_k["gender"], df_k["approved"],
                     rownames=["Gender"], colnames=["Approved"])
chi2_gen, p_gen, dof_gen, _ = chi2_contingency(ct_gen)
print(f"\n  Chi-square test : χ²={chi2_gen:.3f},  df={dof_gen},  p={p_gen:.4f}")
print(f"  Statistically significant (p < 0.05) : {'YES' if p_gen < 0.05 else 'NO'}")
print(f"\nContingency table:")
print(ct_gen)

### 3.2 Age Group — Approval Disparity

In [ ]:
# ── 3.2 Age group approval rates ──────────────────────────────────────────────
df_age = df_k.dropna(subset=["age_group"]).copy()

age_agg = (df_age.groupby("age_group", observed=True)["approved"]
                 .agg(approved_n="sum", total="count", approval_rate="mean")
                 .assign(approval_rate=lambda x: x["approval_rate"].round(4)))

print("Approval Rates by Age Group:")
print(age_agg.to_string())

# Disparate impact: each group relative to highest-approval group
max_rate = age_agg["approval_rate"].max()
print("\n  Disparate Impact Ratios (group ÷ highest-rate group):")
for grp, row in age_agg.iterrows():
    di = row["approval_rate"] / max_rate
    flag = "  ← BELOW 80% threshold" if di < 0.80 else ""
    print(f"    {str(grp):<8}: {di:.4f}{flag}")

# Chi-square test
ct_age = pd.crosstab(df_age["age_group"].astype(str), df_age["approved"],
                     rownames=["Age group"], colnames=["Approved"])
chi2_age, p_age, dof_age, _ = chi2_contingency(ct_age)
print(f"\n  Chi-square test : χ²={chi2_age:.3f},  df={dof_age},  p={p_age:.4f}")
print(f"  Statistically significant (p < 0.05) : {'YES' if p_age < 0.05 else 'NO'}")

In [ ]:
# ── Visualisation: approval rates by gender and age group ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# — Left: Gender —
ax = axes[0]
genders = ["Female", "Male"]
rates   = [gen_agg.loc[g, "approval_rate"] for g in genders]
colours = [COLOUR_F, COLOUR_M]
bars    = ax.bar(genders, rates, color=colours, width=0.45, edgecolor="white", linewidth=1.2)
threshold_line = rate_m * 0.80
ax.axhline(threshold_line, color="#333333", linestyle="--", linewidth=1.2,
           label=f"80% threshold ({threshold_line:.1%})")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Approval Rate", fontsize=11)
ax.set_title("Approval Rate by Gender", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
for bar, r in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{r:.1%}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.text(0.5, 0.04, f"DI = {di_gen:.3f}", ha="center", transform=ax.transAxes,
        fontsize=10, color="#c0392b" if di_gen < 0.80 else "#27ae60",
        fontweight="bold")

# — Right: Age groups —
ax2 = axes[1]
ag_labels = [str(g) for g in age_agg.index]
ag_rates  = age_agg["approval_rate"].values
bar2 = ax2.bar(ag_labels, ag_rates, color=COLOUR_NEU, width=0.45,
               edgecolor="white", linewidth=1.2)
ax2.axhline(max_rate * 0.80, color="#333333", linestyle="--", linewidth=1.2,
            label=f"80% threshold ({max_rate*0.80:.1%})")
ax2.set_ylim(0, 1.0)
ax2.set_ylabel("Approval Rate", fontsize=11)
ax2.set_title("Approval Rate by Age Group", fontsize=12, fontweight="bold")
ax2.set_xlabel("Age group (years at 2024-01-01)", fontsize=10)
ax2.legend(fontsize=9)
for bar, r in zip(bar2, ag_rates):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
             f"{r:.1%}", ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_baseline_disparity.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/fig_baseline_disparity.png")

**Findings — Baseline Disparity**

| | Gender | Age |
|---|---|---|
| **Objective** | Detect raw approval-rate disparity between Male and Female applicants | Detect approval-rate disparity across age cohorts |
| **Method / threshold** | DI (Female ÷ Male); χ² test; DI < 0.80 = adverse impact | DI (group ÷ highest-rate group); χ² test; DI < 0.80 |
| **Results** | Female: ~50.6% · Male: ~66.0% · DI ≈ 0.77 · p < 0.05 | <30 group shows lowest approval rate; DI for <30 < 0.80 |
| **Interpretation** | The gender DI falls **below the 0.80 regulatory threshold**, indicating prima facie adverse impact against female applicants. The disparity is statistically significant. | Applicants under 30 experience disproportionately lower approval rates. The pattern is consistent with age-based disparity and is statistically significant. |

**Risk assessment:**

| Finding | Metric | Threshold | Severity | Governance Impact |
|---|---|---|---|---|
| Female approval rate below 80% of male rate | DI ≈ 0.77 | DI ≥ 0.80 | **High** | Prima facie adverse impact; regulatory review required |
| Age <30 approval rate disproportionately low | DI < 0.80 | DI ≥ 0.80 | **Moderate** | Age discrimination risk; ECOA review warranted |

<a id="conditional"></a>
## 4. Conditional Fairness Analysis

A raw approval-rate disparity does not in itself establish discriminatory intent — legitimate financial risk factors (income, debt-to-income ratio, credit history, savings) may explain part of the gap.
This section uses logistic regression to isolate the gender and age effects **after controlling** for financial variables.
If gender or age remain statistically significant predictors after controls, the fairness risk is elevated from descriptive to conditional — a materially more serious governance finding.

In [ ]:
# ── 4. Conditional fairness — logistic regression ─────────────────────────────
df_model = df_k.copy()

# Encode gender: Male = 1, Female = 0 (binary; reference = Female)
df_model["gender_bin"] = (df_model["gender"] == "Male").astype(int)

FEATURE_COLS = ["gender_bin", "age", "income", "dti", "credit_hist", "savings"]
TARGET       = "approved"

df_model_clean = df_model[FEATURE_COLS + [TARGET]].dropna()
print(f"Model set: {len(df_model_clean)} records  "
      f"(dropped {len(df_model) - len(df_model_clean)} with missing values)")

X       = df_model_clean[FEATURE_COLS]
y       = df_model_clean[TARGET]
X_const = sm.add_constant(X)

logit   = sm.Logit(y, X_const).fit(disp=0)
print()
print(logit.summary())
print()

# Odds ratios
params = logit.params.drop("const")
pvals  = logit.pvalues.drop("const")
ci     = logit.conf_int().drop("const")

or_df = pd.DataFrame({
    "Coeff"         : params.round(4),
    "Odds Ratio"    : np.exp(params).round(4),
    "OR 95% CI low" : np.exp(ci[0]).round(4),
    "OR 95% CI hi"  : np.exp(ci[1]).round(4),
    "p-value"       : pvals.round(4),
    "Significant"   : pvals.lt(0.05).map({True: "YES", False: "no"}),
}).rename_axis("Feature")

print("Odds Ratios (reference: Female, standardised interpretation):")
print(or_df.to_string())

**Findings — Conditional Fairness**

| | Detail |
|---|---|
| **Objective** | Determine whether gender and age disparities persist after controlling for income, DTI, credit history, and savings |
| **Method** | Logistic regression: `approved ~ gender + age + income + dti + credit_hist + savings` |
| **Threshold** | Gender / age coefficient p < 0.05 while controlling for financial risk |
| **Interpretation** | `gender_bin` (Male indicator) is positive and highly significant: **OR = 2.01, p = 0.0003**. Male applicants are twice as likely to be approved as female applicants with identical income, DTI, credit history, and savings profiles. This is the most serious finding in the audit — the disparity is not explained by financial risk. Age is not significant after controls (p = 0.642), suggesting the age gap observed in the baseline is partially attributable to financial risk differences in younger applicants. |

**Risk assessment:**

| Finding | Metric | Threshold | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|---|
| Gender significant after financial controls — OR = 2.01 | p(gender_bin) = 0.0003 | p ≥ 0.05 | **Critical** | Conditional discrimination confirmed; risk-factor explanation ruled out. NovaCred's automated credit decisions constitute profiling with significant legal effects — directly engaging Regulation (EU) 2016/679 (GDPR), Art. 22, which grants data subjects the right not to be subject to solely automated decisions. | Immediate process audit · Legal review · Fairness-aware underwriting redesign |
| Age not significant after financial controls | p(age) = 0.642 | p < 0.05 | **Low** | Age baseline disparity largely explained by financial risk; no conditional age discrimination indicated | Continue monitoring; no immediate conditional action required |

<a id="pricing"></a>
## 5. Pricing Fairness — Interest Rate Analysis

Even where approval rates show adverse impact, pricing may introduce a secondary layer of discrimination. This section examines whether approved applicants face materially different interest rates depending on gender or age.
Analysis is restricted to the 292 approved records where an interest rate was assigned.

In [ ]:
# ── 5.1 Interest rate by gender (approved applicants only) ────────────────────
df_appr = df_k[df_k["approved"] == 1].dropna(subset=["interest_rate"]).copy()

print(f"Approved with interest rate data: {len(df_appr)} records")
print()

rate_summary = (df_appr.groupby("gender")["interest_rate"]
                       .agg(mean="mean", median="median", std="std", n="count")
                       .round(4))
print("Interest Rate by Gender (approved applicants):")
print(rate_summary.to_string())

# Mann-Whitney U test (non-parametric; appropriate for rate distributions)
rates_f = df_appr[df_appr["gender"] == "Female"]["interest_rate"]
rates_m = df_appr[df_appr["gender"] == "Male"  ]["interest_rate"]
mw_stat, p_mw = mannwhitneyu(rates_f, rates_m, alternative="two-sided")
print(f"\nMann-Whitney U test:  stat={mw_stat:.1f},  p={p_mw:.4f}")
print(f"Statistically significant (p < 0.05): {'YES' if p_mw < 0.05 else 'NO'}")

# ── 5.2 Interest rate by age group ────────────────────────────────────────────
print()
df_appr_age = df_appr.dropna(subset=["age_group"])
rate_age = (df_appr_age.groupby("age_group", observed=True)["interest_rate"]
                       .agg(mean="mean", median="median", n="count").round(4))
print("Interest Rate by Age Group (approved applicants):")
print(rate_age.to_string())

In [ ]:
# ── 5.3 OLS regression — interest rate controlled for financial risk ───────────
df_appr["gender_bin"] = (df_appr["gender"] == "Male").astype(int)
FEAT_RATE = ["gender_bin", "age", "income", "dti", "credit_hist", "savings"]

df_rate_clean = df_appr[FEAT_RATE + ["interest_rate"]].dropna()
print(f"OLS model set: {len(df_rate_clean)} records")

Xr = sm.add_constant(df_rate_clean[FEAT_RATE])
yr = df_rate_clean["interest_rate"]
ols = sm.OLS(yr, Xr).fit()
print(ols.summary())

In [ ]:
# ── Visualisation: interest rate distributions by gender ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box plot by gender
ax = axes[0]
data_plot = [rates_f.values, rates_m.values]
bp = ax.boxplot(data_plot, labels=["Female", "Male"], patch_artist=True,
                medianprops=dict(color="white", linewidth=2))
for patch, colour in zip(bp["boxes"], [COLOUR_F, COLOUR_M]):
    patch.set_facecolor(colour)
ax.set_ylabel("Interest Rate (%)", fontsize=11)
ax.set_title("Interest Rate Distribution by Gender\n(Approved applicants)", fontsize=11, fontweight="bold")
ax.text(0.5, 0.02, f"Mann-Whitney p = {p_mw:.4f}", ha="center", transform=ax.transAxes,
        fontsize=9, style="italic")

# Bar chart: mean rate by age group
ax2 = axes[1]
ax2.bar(rate_age.index.astype(str), rate_age["mean"], color=COLOUR_NEU, width=0.45,
        edgecolor="white", linewidth=1.2)
ax2.set_ylabel("Mean Interest Rate (%)", fontsize=11)
ax2.set_title("Mean Interest Rate by Age Group\n(Approved applicants)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Age group (years at 2024-01-01)", fontsize=10)
for i, (grp, row) in enumerate(rate_age.iterrows()):
    ax2.text(i, row["mean"] + 0.03, f"{row['mean']:.2f}%", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_pricing_fairness.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/fig_pricing_fairness.png")

**Findings — Pricing Fairness**

| | Detail |
|---|---|
| **Objective** | Determine whether approved applicants face materially different interest rates by gender or age |
| **Method** | Descriptive comparison + Mann–Whitney U test; OLS regression controlling for financial risk |
| **Threshold** | p < 0.05 · Gender coefficient direction and significance |
| **Interpretation** | Female applicants receive marginally lower interest rates than male applicants among approved loans. This direction is *not adverse* to female applicants in pricing. If the OLS gender coefficient is not significant after controls, pricing disparity is explained by risk profile differences rather than gender. |

**Risk assessment:**

| Finding | Metric | Threshold | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|---|
| Interest rate difference by gender | Mann–Whitney p-value · OLS gender coeff | p < 0.05 | **Low** (if not significant after controls) | Pricing channel does not appear to compound approval disparity | Monitor quarterly; include in annual fairness audit |
| Age-related rate variation | OLS age coeff | p < 0.05 | **Low–Moderate** | Review if older applicants face systematically higher rates | Audit rate-setting model for age dependency |

<a id="proxy"></a>
## 6. Proxy Discrimination Risk

Proxy discrimination occurs when a variable that appears neutral encodes protected-attribute information and is used in a way that produces discriminatory outcomes. Two proxy candidates are evaluated:

1. **ZIP code** — geographic assignment may encode gender or demographic composition
2. **Spending behaviour** — certain spending categories may correlate with demographic characteristics

**Both conditions must hold simultaneously** for proxy discrimination to be established:
- **Condition 1:** The proxy variable is statistically associated with a protected attribute (gender / age)
- **Condition 2:** The proxy variable influences the decision outcome after controlling for legitimate financial risk factors

### 6.1 ZIP Code as Proxy

In [ ]:
# ── 6.1 ZIP code proxy discrimination test ────────────────────────────────────

# Overview: ZIP prefix distribution
print("ZIP-3 prefix distribution:")
print(df_k["zip3"].value_counts().to_string())
print()

# Focus on the two dominant ZIP-3 groups (sufficient sample size)
df_zip = df_k[df_k["zip3"].isin(["100", "902"])].copy()

# CONDITION 1: ZIP ↔ gender association (chi-square)
ct_zip_gender = pd.crosstab(df_zip["zip3"], df_zip["gender"],
                             rownames=["ZIP-3"], colnames=["Gender"])
chi2_zg, p_zg, dof_zg, _ = chi2_contingency(ct_zip_gender)

print("Condition 1 — ZIP vs Gender:")
print(ct_zip_gender.to_string())
print(f"  χ²={chi2_zg:.2f},  df={dof_zg},  p={p_zg:.2e}")
print(f"  Condition 1 met: {'YES — ZIP is strongly associated with gender' if p_zg < 0.05 else 'NO'}")
print()

# Approval rates by ZIP
zip_approval = (df_k.groupby("zip3")["approved"]
                    .agg(approved_n="sum", total="count", approval_rate="mean")
                    .assign(approval_rate=lambda x: x["approval_rate"].round(4)))
print("Approval rates by ZIP-3 prefix:")
print(zip_approval.to_string())
print()

# CONDITION 2: ZIP → approval after financial controls (logistic regression)
df_zip_model = df_k.copy()
df_zip_model["gender_bin"] = (df_zip_model["gender"] == "Male").astype(int)
# ZIP-3 dummy: 100 = 1, 902 = 0  (restrict to dominant ZIPs for clean binary test)
df_zip_model = df_zip_model[df_zip_model["zip3"].isin(["100", "902"])].copy()
df_zip_model["zip_100"]    = (df_zip_model["zip3"] == "100").astype(int)

FEAT_ZIP   = ["gender_bin", "age", "income", "dti", "credit_hist", "savings", "zip_100"]
df_zip_cln = df_zip_model[FEAT_ZIP + ["approved"]].dropna()

Xz      = sm.add_constant(df_zip_cln[FEAT_ZIP])
yz      = df_zip_cln["approved"]
logit_z = sm.Logit(yz, Xz).fit(disp=0)

coef_zip = logit_z.params["zip_100"]
p_zip    = logit_z.pvalues["zip_100"]
print("Condition 2 — ZIP effect on approval (controlling for financial risk + gender):")
print(f"  zip_100 coefficient : {coef_zip:.4f}")
print(f"  Odds ratio          : {np.exp(coef_zip):.4f}")
print(f"  p-value             : {p_zip:.4f}")
print(f"  Condition 2 met: {'YES — ZIP is a significant predictor after controls' if p_zip < 0.05 else 'NO'}")
print()
both_zip = p_zg < 0.05 and p_zip < 0.05
print(f"PROXY DISCRIMINATION VERDICT (ZIP): {'Both conditions met — proxy risk confirmed' if both_zip else 'Not fully established'}")

### 6.2 Spending Behaviour as Proxy

Beyond its role as a potential proxy variable, the presence of sensitive spending categories (Gambling, Alcohol, Adult Entertainment) in the dataset raises an independent governance concern. The collection of behavioural data reflecting personal lifestyle choices requires a clearly documented necessity for the credit decision process. Where that necessity cannot be demonstrated, retaining such data conflicts with the principle of data minimisation under Regulation (EU) 2016/679 (GDPR), Art. 5(1)(c). This concern is assessed separately from the statistical proxy test below.

In [ ]:
# ── 6.2 Spending behaviour proxy test ─────────────────────────────────────────
# High-risk spending flag: Gambling, Alcohol, Adult Entertainment

flagged_counts = {
    "Gambling"         : df_k["spend_gambling"].notna().sum()         if "spend_gambling" in df_k.columns else 0,
    "Alcohol"          : df_k["spend_alcohol"].notna().sum()           if "spend_alcohol" in df_k.columns else 0,
    "Adult Entertain." : df_k["spend_adult_entertainment"].notna().sum() if "spend_adult_entertainment" in df_k.columns else 0,
}
print("Flagged spending category counts (records with any amount):")
for cat, cnt in flagged_counts.items():
    print(f"  {cat:<22}: {cnt}")

total_flagged = df_k["spend_flagged"].sum()
print(f"\nRecords with any flagged category: {total_flagged} ({total_flagged/len(df_k)*100:.1f}%)")
print()

# CONDITION 1: flagged spending ↔ gender
ct_spend_gender = pd.crosstab(df_k["spend_flagged"], df_k["gender"],
                               rownames=["Flagged spending"], colnames=["Gender"])
print("Condition 1 — Flagged Spending vs Gender:")
print(ct_spend_gender.to_string())

if ct_spend_gender.values.min() >= 5:
    chi2_sg, p_sg, _, _ = chi2_contingency(ct_spend_gender)
    print(f"  χ²={chi2_sg:.4f},  p={p_sg:.4f}")
    print(f"  Condition 1 met: {'YES' if p_sg < 0.05 else 'NO'}")
else:
    print("  NOTE: Expected cell counts < 5 — chi-square result unreliable due to small sample.")
    p_sg = 1.0
print()

# CONDITION 2: flagged spending → approval after controls
df_sp_model = df_k.copy()
df_sp_model["gender_bin"] = (df_sp_model["gender"] == "Male").astype(int)
FEAT_SP    = ["gender_bin", "age", "income", "dti", "credit_hist", "savings", "spend_flagged"]
df_sp_cln  = df_sp_model[FEAT_SP + ["approved"]].dropna()

Xs      = sm.add_constant(df_sp_cln[FEAT_SP])
ys      = df_sp_cln["approved"]
logit_s = sm.Logit(ys, Xs).fit(disp=0)

coef_sp = logit_s.params["spend_flagged"]
p_sp    = logit_s.pvalues["spend_flagged"]
print("Condition 2 — Flagged spending effect on approval (controlling for financial risk + gender):")
print(f"  spend_flagged coefficient : {coef_sp:.4f}")
print(f"  Odds ratio                : {np.exp(coef_sp):.4f}")
print(f"  p-value                   : {p_sp:.4f}")
print(f"  Condition 2 met: {'YES' if p_sp < 0.05 else 'NO'}")
print()
both_sp = p_sg < 0.05 and p_sp < 0.05
print(f"PROXY DISCRIMINATION VERDICT (Spending): {'Both conditions met — proxy risk confirmed' if both_sp else 'Not fully established (insufficient evidence or sample size)'}")

**Findings — Proxy Discrimination**

| | ZIP Code | Spending Behaviour |
|---|---|---|
| **Objective** | Determine whether ZIP code encodes gender and independently influences approval | Determine whether spending categories encode gender and influence approval |
| **Method** | χ² (ZIP ↔ gender) + logistic regression (ZIP → approval after controls) | χ² (flagged spending ↔ gender) + logistic regression |
| **Condition 1** | ZIP-3 `100` is 88% Male; ZIP-3 `902` is 94% Female — χ² highly significant | Gambling / Alcohol / Adult Entertainment counts are very small (< 20 records each) — statistical reliability limited |
| **Condition 2** | **Not confirmed** — ZIP-100 coefficient in logistic model not significant after controls (p = 0.842) | Not confirmed — flagged spending p = 0.576 |
| **Interpretation** | ZIP code is a near-perfect gender proxy (Condition 1 strongly confirmed). However, ZIP was not found to independently predict approval after controlling for financial risk and gender (Condition 2 not confirmed). Full proxy discrimination is not established, but the structural correlation creates latent risk if the model specification changes. | Sample sizes for high-risk spending categories are too small (n < 20) for reliable inference. Neither condition was confirmed. |

**Risk assessment:**

| Finding | Metric | Threshold | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|---|
| ZIP code near-perfectly correlated with gender | χ² p < 0.001 | Independent of outcome | **High** | If ZIP influences decisions, it constitutes indirect gender discrimination. Furthermore, retaining a geographic field that encodes protected-attribute information without demonstrated necessity for the credit decision raises a data minimisation concern under Regulation (EU) 2016/679 (GDPR), Art. 5(1)(c), irrespective of whether proxy discrimination is statistically confirmed. | Remove ZIP from all decision inputs · Document lawful basis for retaining geographic data |
| Spending proxy — insufficient data for proxy test | n < 20 per category | — | **Low (unresolved)** | Statistical proxy risk cannot be confirmed or excluded with current sample sizes | Re-test when per-category n ≥ 30 |
| Sensitive spending categories — data collection justification | Gambling · Alcohol · Adult Entertainment present in dataset | Demonstrated necessity for credit decision | **Moderate** | Collecting behavioural data of this nature without a clear, documented link to credit risk assessment is difficult to reconcile with the data minimisation principle under Regulation (EU) 2016/679 (GDPR), Art. 5(1)(c). Spending data touching on lifestyle and personal behaviour may also attract scrutiny under Art. 9 if it is construed as revealing health-related or other sensitive personal characteristics. | Data Protection Officer review required · Document lawful basis and necessity for each spending category · Remove categories that cannot be justified |

<a id="intersectional"></a>
## 7. Intersectional Effects — Gender × Age Group

Intersectional analysis examines whether the combination of two protected attributes produces disproportionate outcomes beyond what either dimension alone would predict. A sub-group that is female *and* young, for example, may face compounded disadvantage.

In [ ]:
# ── 7. Intersectional effects: Gender × Age group ─────────────────────────────
df_inter = df_k.dropna(subset=["age_group"]).copy()

inter_agg = (df_inter.groupby(["gender", "age_group"], observed=True)["approved"]
                     .agg(approved_n="sum", total="count", approval_rate="mean")
                     .assign(approval_rate=lambda x: x["approval_rate"].round(4))
                     .reset_index())

print("Approval rates by Gender × Age Group:")
pivot = inter_agg.pivot(index="age_group", columns="gender",
                         values="approval_rate").round(4)
print(pivot.to_string())
print()

# Disparate impact: Female/Male within each age group
print("Disparate Impact (Female ÷ Male) by Age Group:")
for ag in df_inter["age_group"].cat.categories:
    sub = inter_agg[inter_agg["age_group"] == ag]
    f_r = sub[sub["gender"] == "Female"]["approval_rate"].values
    m_r = sub[sub["gender"] == "Male"  ]["approval_rate"].values
    if len(f_r) and len(m_r) and m_r[0] > 0:
        di = f_r[0] / m_r[0]
        flag = "  ← BELOW 80% threshold" if di < 0.80 else ""
        print(f"  {str(ag):<8}: DI = {di:.4f}{flag}")
print()

# Chi-square on full gender × age interaction
df_inter["gender_age"] = df_inter["gender"] + " · " + df_inter["age_group"].astype(str)
ct_inter = pd.crosstab(df_inter["gender_age"], df_inter["approved"],
                        rownames=["Gender × Age"], colnames=["Approved"])
chi2_inter, p_inter, dof_inter, _ = chi2_contingency(ct_inter)
print(f"Chi-square (Gender × Age group vs Approved): χ²={chi2_inter:.3f},  df={dof_inter},  p={p_inter:.4f}")
print(f"Statistically significant: {'YES' if p_inter < 0.05 else 'NO'}")

In [ ]:
# ── Visualisation: intersectional heatmap ─────────────────────────────────────
pivot_plot = inter_agg.pivot(index="age_group", columns="gender", values="approval_rate")

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot_plot.values, cmap="RdYlGn", vmin=0.20, vmax=0.90, aspect="auto")

ax.set_xticks(range(len(pivot_plot.columns)))
ax.set_xticklabels(pivot_plot.columns, fontsize=11)
ax.set_yticks(range(len(pivot_plot.index)))
ax.set_yticklabels([str(g) for g in pivot_plot.index], fontsize=11)
ax.set_xlabel("Gender", fontsize=11)
ax.set_ylabel("Age group (years at 2024-01-01)", fontsize=11)
ax.set_title("Approval Rate — Gender × Age Group Heatmap\n(Green = higher rate, Red = lower rate)",
             fontsize=11, fontweight="bold")

for i in range(pivot_plot.shape[0]):
    for j in range(pivot_plot.shape[1]):
        val = pivot_plot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1%}", ha="center", va="center",
                    fontsize=12, fontweight="bold",
                    color="white" if val < 0.45 or val > 0.75 else "black")

plt.colorbar(im, ax=ax, label="Approval Rate")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_intersectional.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/fig_intersectional.png")

**Findings — Intersectional Effects**

| | Detail |
|---|---|
| **Objective** | Determine whether combinations of gender and age produce compounded approval-rate disparities |
| **Method** | Gender × Age group approval rates; within-group DI; chi-square on combined interaction |
| **Threshold** | DI < 0.80 within any sub-group |
| **Interpretation** | Young female applicants (Female · <30) represent the highest-risk intersectional group. If the DI for this sub-group falls below 0.80, they experience a compounded disadvantage that is not captured by either gender or age analysis alone. The intersection of two protected attributes can amplify disparate impact materially. |

**Risk assessment:**

| Finding | Metric | Threshold | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|---|
| Female · <30 approval rate disproportionately low | DI (Female <30 ÷ Male <30) | DI ≥ 0.80 | **High** (if DI < 0.80) | Compounded intersectional disadvantage | Disaggregated fairness monitoring by gender–age cell |
| Overall gender × age interaction significant | χ² p < 0.05 | — | **Moderate** | Structural interaction warrants periodic audit | Annual intersectional fairness review |

<a id="fairlearn"></a>
## 8. Fairness Metrics Summary — Fairlearn MetricFrame

The Fairlearn `MetricFrame` provides a standardised, reportable fairness metrics table suitable for governance documentation. For this historical audit, the historical decisions are treated as the outcome to be evaluated — no trained prediction model is assessed.

**Note on Equal Opportunity Difference (EOD):**  
EOD measures whether the true positive rate (approval of applicants who *would* repay) is equal across groups. Computing it requires a ground-truth label for actual creditworthiness — i.e., whether each applicant would have repaid the loan. This information does not exist in the NovaCred dataset, which records only the historical decision, not the repayment outcome. EOD is therefore not computable from this data and is explicitly excluded rather than omitted. Demographic Parity Difference and the Disparate Impact Ratio are the appropriate metrics for auditing historical decision outcomes without ground-truth repayment data.

In [ ]:
# ── 8. Fairlearn MetricFrame ──────────────────────────────────────────────────

# ── 8.1 Gender ────────────────────────────────────────────────────────────────
y_hist_gen   = df_k["approved"].values.astype(int)
sf_gender    = df_k["gender"].values

mf_gender = MetricFrame(
    metrics        = {"Approval Rate": selection_rate},
    y_true         = y_hist_gen,
    y_pred         = y_hist_gen,          # historical audit: decision = outcome
    sensitive_features = sf_gender,
)

print("── Fairlearn MetricFrame: Gender ──────────────────────────────────────")
print(f"  Overall approval rate          : {mf_gender.overall['Approval Rate']:.4f}")
print()
print("  By group:")
print(mf_gender.by_group.round(4).to_string())
print()
rates_gender   = mf_gender.by_group["Approval Rate"]
dpd_gender     = mf_gender.difference()["Approval Rate"]
di_fl_gender   = rates_gender.min() / rates_gender.max()
print(f"  Demographic Parity Difference  : {dpd_gender:.4f}")
print(f"  Disparate Impact Ratio         : {di_fl_gender:.4f}  "
      f"({'BELOW 0.80 — adverse impact' if di_fl_gender < 0.80 else 'above 0.80'})")

# ── 8.2 Age group ─────────────────────────────────────────────────────────────
df_fl_age    = df_k.dropna(subset=["age_group"]).copy()
y_hist_age   = df_fl_age["approved"].values.astype(int)
sf_age       = df_fl_age["age_group"].astype(str).values

mf_age = MetricFrame(
    metrics            = {"Approval Rate": selection_rate},
    y_true             = y_hist_age,
    y_pred             = y_hist_age,
    sensitive_features = sf_age,
)

print()
print("── Fairlearn MetricFrame: Age Group ───────────────────────────────────")
print(f"  Overall approval rate          : {mf_age.overall['Approval Rate']:.4f}")
print()
print("  By group:")
print(mf_age.by_group.round(4).to_string())
rates_age   = mf_age.by_group["Approval Rate"]
dpd_age     = mf_age.difference()["Approval Rate"]
di_fl_age   = rates_age.min() / rates_age.max()
print(f"  Demographic Parity Difference  : {dpd_age:.4f}")
print(f"  Disparate Impact Ratio         : {di_fl_age:.4f}  "
      f"({'BELOW 0.80 — adverse impact' if di_fl_age < 0.80 else 'above 0.80'})")

<a id="risk"></a>
## 9. Consolidated Governance Risk Assessment

All findings from Sections 3–8 are consolidated into a single risk matrix using standardised governance severity classifications.

**Severity scale:**
- **Low** — small differences, no threshold crossed
- **Moderate** — statistically significant but limited materiality
- **High** — regulatory threshold violated (DI < 0.80)
- **Critical** — disparity persists after controlling for financial risk variables

In [ ]:
# ── 9. Consolidated risk matrix ───────────────────────────────────────────────
risk_rows = [
    ("Baseline — Gender",
     "Female approval rate below 80% of male rate",
     f"DI ≈ {di_gen:.3f}",
     "DI ≥ 0.80",
     "High",
     "Prima facie adverse impact; ECOA / EU credit regulation review required",
     "Fairness constraint in underwriting · Disparate impact testing"),

    ("Baseline — Age",
     "Applicants under 30 disproportionately rejected",
     f"DI < 0.80 for <30 group",
     "DI ≥ 0.80",
     "Moderate",
     "Age discrimination risk; review underwriting criteria for age dependency",
     "Age-stratified approval monitoring · ECOA age-category audit"),

    ("Conditional — Gender",
     "Gender remains significant after controlling for income, DTI, credit history, savings",
     "Logistic regression: OR=2.01, p=0.0003",
     "p ≥ 0.05 = no residual effect",
     "Critical",
     "Male applicants are twice as likely to be approved as female applicants with identical financial profiles — disparity cannot be explained by risk factors alone",
     "Immediate process audit · Fairness-aware underwriting redesign · Legal review"),

    ("Conditional — Age",
     "Age not significant after financial controls",
     "Logistic regression: p(age)=0.642",
     "p < 0.05",
     "Low",
     "Age disparity in baseline is largely explained by financial risk factors — age itself not an independent predictor",
     "Continue baseline age monitoring; no immediate conditional action required"),

    ("Pricing — Gender",
     "Interest rate difference by gender among approved applicants",
     "Mann-Whitney p-value · OLS gender coeff",
     "p < 0.05",
     "Low (direction not adverse to females)",
     "Pricing channel does not compound approval disparity",
     "Monitor quarterly in annual fairness audit"),

    ("Proxy — ZIP code",
     "ZIP-3 near-perfectly correlated with gender (Condition 1 confirmed); ZIP not independently significant in approval model (Condition 2 not confirmed)",
     "Condition 1: χ² p=8e-72 · Condition 2: logistic p(ZIP)=0.842",
     "Both conditions p < 0.05",
     "Moderate",
     "Structural data concern: ZIP encodes gender information. Current model does not appear to use ZIP as a driver, but its presence creates latent proxy risk if model specifications change",
     "Exclude ZIP from any future automated decision model · Flag in data governance inventory as PII-adjacent proxy variable"),

    ("Proxy — Spending",
     "High-risk spending categories insufficient to test reliably",
     "n < 20 per category",
     "n ≥ 30 per category",
     "Low (unresolved)",
     "Proxy risk cannot be confirmed or excluded with current data volume",
     "Re-test when sample size ≥ 30 per category"),

    ("Intersectional — Female · <30 and Female · 40-49",
     "Female applicants under 30 face severely compounded disadvantage (DI=0.591); Female 40-49 also below threshold (DI=0.780)",
     "DI(Female <30 ÷ Male <30) = 0.591 · DI(Female 40-49 ÷ Male 40-49) = 0.780",
     "DI ≥ 0.80",
     "High",
     "Two intersectional sub-groups fall below the 0.80 threshold — compounded gender–age disadvantage not captured by single-attribute analysis",
     "Disaggregated monitoring by gender–age cell · Priority fairness review for Female <30 and Female 40-49 cohorts"),
]

COLS = ["Domain", "Finding", "Evidence", "Threshold", "Severity", "Governance Impact", "Recommended Control"]
risk_df = pd.DataFrame(risk_rows, columns=COLS)

SEV_ORDER = {"Critical": 0, "High": 1, "Moderate": 2, "Low (unresolved)": 3, "Low": 4}
risk_df["_sev_order"] = risk_df["Severity"].map(lambda s: SEV_ORDER.get(s.split(" ")[0], 9))
risk_df = risk_df.sort_values("_sev_order").drop(columns="_sev_order").reset_index(drop=True)

print("═" * 110)
print("  CONSOLIDATED GOVERNANCE RISK MATRIX — NovaCred Bias & Fairness Audit")
print("═" * 110)
for _, row in risk_df.iterrows():
    print(f"  Domain   : {row['Domain']}")
    print(f"  Finding  : {row['Finding']}")
    print(f"  Evidence : {row['Evidence']}  (threshold: {row['Threshold']})")
    print(f"  Severity : {row['Severity']}")
    print(f"  Impact   : {row['Governance Impact']}")
    print(f"  Control  : {row['Recommended Control']}")
    print("─" * 110)

<a id="executive"></a>
## 10. Executive Summary

### Overall Fairness Risk Classification: **Elevated–High**

---

### Key Findings

**1. Gender — Critical fairness risk**
Female applicants are approved at 50.6% vs. 66.0% for male applicants — a gap of **15.4 percentage points**. The Disparate Impact Ratio of **0.767 falls below the 0.80 regulatory threshold** (χ² p = 0.0007), constituting prima facie adverse impact. Critically, logistic regression confirms that **gender remains a highly significant predictor after controlling for income, DTI, credit history, and savings** (OR = 2.01, p = 0.0003). Male applicants are twice as likely to be approved as female applicants with an identical financial profile. This constitutes a **Critical** governance finding: the disparity cannot be explained by legitimate risk factors.

**2. Age — Moderate baseline risk, not confirmed conditionally**
Applicants under 30 face the lowest approval rate (41.4%, DI = 0.601 vs. the highest-rate group). The baseline disparity is statistically significant (χ² p < 0.001). However, after controlling for financial risk factors, age is not independently significant (logistic p = 0.642), suggesting the age gap is partially explained by younger applicants having weaker financial profiles rather than age-based discrimination per se.

**3. ZIP code — Moderate latent proxy risk**
ZIP-3 prefix is near-perfectly correlated with gender (χ² p = 8×10⁻⁷²): ZIP 100 is 88% Male, ZIP 902 is 94% Female. However, ZIP was not found to independently influence approval after controlling for financial risk and gender (logistic p = 0.842), so full proxy discrimination cannot be established with current data. The structural correlation remains a latent risk that must be managed.

**4. Pricing — Low risk**
No statistically significant difference in interest rates by gender was found among approved applicants (Mann–Whitney p = 0.326; OLS gender coefficient p = 0.298). The pricing channel does not compound the approval-stage disparity.

**5. Intersectional — High risk in two sub-groups**
Female applicants under 30 (DI = 0.591) and female applicants aged 40–49 (DI = 0.780) both fall below the 0.80 threshold. Young female applicants are approved at less than 60% of the rate of equivalent male peers — the most severely disadvantaged intersectional sub-group in the dataset.

---

### Governance Actions Required

| Priority | Action |
|---|---|
| **Immediate** | Legal review of gender-based approval gap — conditional disparity (OR = 2.01, p = 0.0003) constitutes a Critical risk |
| **Immediate** | Audit underwriting criteria for any gender-correlated decision rules |
| **Short-term** | Exclude ZIP code from all automated decision inputs — near-perfect gender proxy |
| **Short-term** | Deploy disaggregated fairness monitoring dashboards by gender × age sub-group |
| **Short-term** | Implement fairness constraints in any future automated underwriting model |
| **Ongoing** | Annual bias audits covering all three fairness pillars |
| **Ongoing** | Re-test spending behaviour proxy when per-category sample size reaches n ≥ 30 |